# Add Hebrew Line Names to Hubs Data

This notebook adds Hebrew line names to the hubs_with_complete_data output.

**Input files:**
1. Output from COMPLETE_TRANSIT_PIPELINE.ipynb (e.g., `hubs_with_complete_data.csv` or `grouped_hubs_ready_for_scoring_*.csv`)
2. CSV file with columns:
   - `LineName`: Line ID (matches values in `Line_Unique` column)
   - `Line_n_Mode`: Full Hebrew name for the line

**Process:**
1. Load the hubs data
2. Parse `Line_Unique` column from string format to actual Python list
3. Load Hebrew line names CSV
4. Create a mapping dictionary: LineName → Line_n_Mode
5. For each hub, map all line IDs to their Hebrew names
6. Create new column `Line_Hebrew_Names` with list of Hebrew names

**Output:**
- Updated CSV with new `Line_Hebrew_Names` column

## Setup

In [ ]:
import pandas as pd
import numpy as np
import ast
import re
from pathlib import Path
from IPython.display import display

# Setup paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
RESULTS_DIR = DATA_DIR / "results"

# Create directories if needed
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Results directory: {RESULTS_DIR}")

## Step 1: Load Hubs Data

Load the output from COMPLETE_TRANSIT_PIPELINE which contains the `Line_Unique` column.

In [ ]:
# Configure input file path
# Option 1: Provide full path to your hubs file
hubs_file = NOTEBOOKS_DIR / 'grouped_hubs_ready_for_scoring_21082025.csv'

# Option 2: Or specify a different file
# hubs_file = RESULTS_DIR / 'hubs_with_complete_data.csv'

# Check if file exists
if not hubs_file.exists():
    print(f"ERROR: File not found: {hubs_file}")
    print("\nPlease update the 'hubs_file' variable above with the correct path.")
    raise FileNotFoundError(f"Hubs file not found: {hubs_file}")

print(f"Loading hubs data from: {hubs_file.name}")
hubs_df = pd.read_csv(hubs_file, encoding='utf-8')

print(f"\nLoaded {len(hubs_df)} hubs")
print(f"Columns: {len(hubs_df.columns)}")

# Check for required columns
if 'Line_Unique' not in hubs_df.columns:
    raise ValueError("'Line_Unique' column not found in hubs data")

print("\n✓ Required column 'Line_Unique' found")

# Display sample
print("\nSample of Line_Unique column:")
display(hubs_df[['group', 'Line_Unique']].head(10))

## Step 2: Parse Line_Unique Column

The `Line_Unique` column is stored as a string representation of a list.
We need to convert it to an actual Python list.

Examples:
- `"['rail_1_1', 'rail_1_2']"` → `['rail_1_1', 'rail_1_2']`
- `['brt022']` → `['brt022']`

In [ ]:
def parse_line_unique(value):
    """
    Parse Line_Unique column from string format to list.
    
    Handles multiple formats:
    - "['line1', 'line2']" (string with quotes)
    - ['line1', 'line2'] (already a list)
    - ["['line1' 'line2']"] (nested string)
    
    Args:
        value: Raw value from Line_Unique column
    
    Returns:
        list: List of line IDs
    """
    if pd.isna(value) or value == '' or value == '[]':
        return []
    
    # If already a list, return it
    if isinstance(value, list):
        return value
    
    # Convert to string
    value = str(value).strip()
    
    # Try using ast.literal_eval (safest method)
    try:
        result = ast.literal_eval(value)
        if isinstance(result, list):
            return result
    except (ValueError, SyntaxError):
        pass
    
    # Fallback: Manual parsing
    # Remove outer brackets and quotes
    value = value.strip('[]"\'')
    
    # Check if it's a nested structure like "['item1' 'item2']"
    nested_pattern = r"\[([^\]]+)\]"
    nested_match = re.search(nested_pattern, value)
    if nested_match:
        inner_content = nested_match.group(1)
        # Remove quotes and split by space or comma
        inner_content = inner_content.replace("'", "").replace('"', '')
        lines = re.split(r'[,\s]+', inner_content)
        lines = [x.strip() for x in lines if x.strip()]
        return lines
    
    # Split by comma
    if ',' in value:
        lines = [x.strip().strip("'\" ") for x in value.split(',')]
        return [x for x in lines if x]
    
    # Single value
    value = value.strip("'\" ")
    return [value] if value else []

# Test the parser with sample data
print("Testing parser with sample Line_Unique values:")
print("" + "="*80)
for idx in range(min(10, len(hubs_df))):
    val = hubs_df.loc[idx, 'Line_Unique']
    parsed = parse_line_unique(val)
    print(f"\nRow {idx}:")
    print(f"  Original: {val}")
    print(f"  Parsed: {parsed}")
    print(f"  Count: {len(parsed)}")

In [ ]:
# Apply parser to entire column
print("Parsing Line_Unique column for all hubs...")
hubs_df['Line_Unique_List'] = hubs_df['Line_Unique'].apply(parse_line_unique)

# Statistics
total_lines = hubs_df['Line_Unique_List'].apply(len).sum()
unique_lines = set()
for lines in hubs_df['Line_Unique_List']:
    unique_lines.update(lines)

print(f"\n✓ Parsing complete!")
print(f"  Total line entries: {total_lines}")
print(f"  Unique line IDs: {len(unique_lines)}")
print(f"  Lines per hub (avg): {total_lines / len(hubs_df):.2f}")

# Show distribution
lines_per_hub = hubs_df['Line_Unique_List'].apply(len)
print(f"\nLines per hub distribution:")
print(f"  Min: {lines_per_hub.min()}")
print(f"  Max: {lines_per_hub.max()}")
print(f"  Median: {lines_per_hub.median():.0f}")
print(f"  Mean: {lines_per_hub.mean():.2f}")

# Display sample
print("\nSample of parsed data:")
display(hubs_df[['group', 'Line_Unique', 'Line_Unique_List']].head(10))

## Step 3: Load Hebrew Line Names CSV

**Expected CSV format:**
- `LineName`: Line ID (e.g., 'rail_1_1', 'brt022')
- `Line_n_Mode`: Full Hebrew name for the line

**IMPORTANT:** Please provide the path to your Hebrew line names CSV file below.

In [ ]:
# ==========================================
# CONFIGURE THIS: Path to Hebrew line names CSV
# ==========================================

# Option 1: Place file in data directory
hebrew_names_file = DATA_DIR / 'hebrew_line_names.csv'

# Option 2: Or specify full path
# hebrew_names_file = Path('/path/to/your/hebrew_line_names.csv')

# ==========================================

# Check if file exists
if not hebrew_names_file.exists():
    print(f"ERROR: Hebrew line names file not found!")
    print(f"Expected location: {hebrew_names_file}")
    print(f"\nPlease provide a CSV file with these columns:")
    print(f"  - LineName: Line ID (e.g., 'rail_1_1', 'brt022')")
    print(f"  - Line_n_Mode: Full Hebrew name")
    print(f"\nUpdate the 'hebrew_names_file' variable in the cell above.")
    raise FileNotFoundError(f"Hebrew line names file not found: {hebrew_names_file}")

print(f"Loading Hebrew line names from: {hebrew_names_file.name}")
hebrew_names_df = pd.read_csv(hebrew_names_file, encoding='utf-8')

print(f"\nLoaded {len(hebrew_names_df)} line name records")
print(f"Columns: {hebrew_names_df.columns.tolist()}")

# Check for required columns
if 'LineName' not in hebrew_names_df.columns:
    raise ValueError("'LineName' column not found in Hebrew names file")
if 'Line_n_Mode' not in hebrew_names_df.columns:
    raise ValueError("'Line_n_Mode' column not found in Hebrew names file")

print("\n✓ Required columns found")

# Display sample
print("\nSample of Hebrew line names:")
display(hebrew_names_df[['LineName', 'Line_n_Mode']].head(20))

## Step 4: Create Line Name Mapping

Create a dictionary that maps LineName → Line_n_Mode (Hebrew name)

In [ ]:
# Clean line names (remove extra whitespace, etc.)
def clean_line_name(name):
    """Clean line name: strip whitespace, normalize."""
    if pd.isna(name):
        return name
    return str(name).strip()

hebrew_names_df['LineName'] = hebrew_names_df['LineName'].apply(clean_line_name)
hebrew_names_df['Line_n_Mode'] = hebrew_names_df['Line_n_Mode'].apply(clean_line_name)

# Remove any duplicates (keep first occurrence)
hebrew_names_df_unique = hebrew_names_df.drop_duplicates(subset='LineName', keep='first')

if len(hebrew_names_df_unique) < len(hebrew_names_df):
    print(f"⚠️  Warning: Found {len(hebrew_names_df) - len(hebrew_names_df_unique)} duplicate LineName entries")
    print(f"   Keeping first occurrence for each line.")

# Create mapping dictionary
line_name_mapping = dict(zip(
    hebrew_names_df_unique['LineName'],
    hebrew_names_df_unique['Line_n_Mode']
))

print(f"\n✓ Created mapping for {len(line_name_mapping)} unique lines")

# Show sample of mapping
print("\nSample of line name mapping:")
for i, (line_id, hebrew_name) in enumerate(list(line_name_mapping.items())[:10]):
    print(f"  {line_id} → {hebrew_name}")

## Step 5: Map Line IDs to Hebrew Names

For each hub, convert the list of line IDs to a list of Hebrew names.

In [ ]:
def map_lines_to_hebrew(line_ids, mapping):
    """
    Map a list of line IDs to their Hebrew names.
    
    Args:
        line_ids: List of line IDs
        mapping: Dictionary mapping line ID to Hebrew name
    
    Returns:
        list: List of Hebrew names (same order as input)
              For unmatched IDs, returns the original ID with "[Unknown]" prefix
    """
    if not line_ids:
        return []
    
    hebrew_names = []
    for line_id in line_ids:
        # Clean the line ID
        line_id_clean = clean_line_name(line_id)
        
        # Look up Hebrew name
        if line_id_clean in mapping:
            hebrew_names.append(mapping[line_id_clean])
        else:
            # If not found, use original ID with warning marker
            hebrew_names.append(f"[Unknown] {line_id_clean}")
    
    return hebrew_names

# Apply mapping to all hubs
print("Mapping line IDs to Hebrew names...")
hubs_df['Line_Hebrew_Names'] = hubs_df['Line_Unique_List'].apply(
    lambda x: map_lines_to_hebrew(x, line_name_mapping)
)

print("\n✓ Mapping complete!")

# Check for unmatched lines
all_matched_names = []
for names in hubs_df['Line_Hebrew_Names']:
    all_matched_names.extend(names)

unmatched = [name for name in all_matched_names if name.startswith('[Unknown]')]
unmatched_unique = set(unmatched)

if unmatched_unique:
    print(f"\n⚠️  Warning: {len(unmatched_unique)} unique line IDs could not be matched:")
    for name in sorted(unmatched_unique)[:20]:
        print(f"     - {name}")
    if len(unmatched_unique) > 20:
        print(f"     ... and {len(unmatched_unique) - 20} more")
    print(f"\n   Total unmatched occurrences: {len(unmatched)}")
else:
    print("\n✅ All line IDs successfully matched to Hebrew names!")

# Display sample
print("\nSample of results:")
display(hubs_df[['group', 'Line_Unique_List', 'Line_Hebrew_Names']].head(10))

## Step 6: Verify Results

Show detailed examples of the mapping.

In [ ]:
# Show detailed examples
print("Detailed mapping examples:")
print("="*80)

for idx in range(min(5, len(hubs_df))):
    group = hubs_df.loc[idx, 'group']
    line_ids = hubs_df.loc[idx, 'Line_Unique_List']
    hebrew_names = hubs_df.loc[idx, 'Line_Hebrew_Names']
    
    print(f"\nGroup {group}:")
    print(f"  Number of lines: {len(line_ids)}")
    print(f"  Line IDs → Hebrew Names:")
    for line_id, hebrew_name in zip(line_ids, hebrew_names):
        print(f"    {line_id} → {hebrew_name}")

# Statistics
total_hebrew_names = hubs_df['Line_Hebrew_Names'].apply(len).sum()
avg_per_hub = total_hebrew_names / len(hubs_df)

print(f"\n" + "="*80)
print(f"Summary Statistics:")
print(f"  Total hubs: {len(hubs_df)}")
print(f"  Total line-hub mappings: {total_hebrew_names}")
print(f"  Average Hebrew names per hub: {avg_per_hub:.2f}")
print(f"  Hubs with at least one line: {(hubs_df['Line_Hebrew_Names'].apply(len) > 0).sum()}")

## Step 7: Export Results

Save the updated dataframe with the new `Line_Hebrew_Names` column.

In [ ]:
# Create output filename
timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M')
output_file = RESULTS_DIR / f'hubs_with_hebrew_line_names_{timestamp}.csv'

# Select columns to export
# Keep original columns plus new ones
export_df = hubs_df.copy()

# Convert Line_Hebrew_Names list to string for CSV export
# (Keep the list format readable)
export_df['Line_Hebrew_Names'] = export_df['Line_Hebrew_Names'].apply(str)

# Optionally, also keep Line_Unique_List as string
if 'Line_Unique_List' in export_df.columns:
    export_df['Line_Unique_List'] = export_df['Line_Unique_List'].apply(str)

# Save to CSV
export_df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"✓ Results exported to: {output_file.name}")
print(f"\nFull path: {output_file}")
print(f"\nExport summary:")
print(f"  Total hubs: {len(export_df)}")
print(f"  Total columns: {len(export_df.columns)}")
print(f"\nNew columns added:")
print(f"  - Line_Unique_List (parsed list from Line_Unique)")
print(f"  - Line_Hebrew_Names (Hebrew names for all lines)")

## Summary

✓ Processing complete!

### What was done:

1. **Loaded hubs data** with `Line_Unique` column
2. **Parsed Line_Unique** from string format to Python list
3. **Loaded Hebrew line names** mapping (LineName → Line_n_Mode)
4. **Mapped line IDs** to their Hebrew names
5. **Created new column** `Line_Hebrew_Names` with list of Hebrew names
6. **Exported results** to CSV file

### Output file:

- Location: `data/results/hubs_with_hebrew_line_names_YYYYMMDD_HHMM.csv`
- Contains all original columns plus:
  - `Line_Unique_List`: Parsed list of line IDs
  - `Line_Hebrew_Names`: List of Hebrew names for all lines

### Next steps:

- Review the output file to verify Hebrew names are correct
- If there were unmatched lines (marked with "[Unknown]"), add them to the Hebrew names CSV and re-run
- Use the `Line_Hebrew_Names` column for visualization or reporting